# Chapter 9: Samples in Structural Engineering
## Using Data to Understand Structural Loads

**Learning Objectives:** By the end of this chapter, you will be able to:
- Explain why engineers must *sample* structural data rather than measure everything
- Identify how sampling bias can lead to dangerous underestimates of structural loads
- Describe how sample size affects the reliability of load estimates
- Apply sampling vocabulary — population, parameter, statistic, SRS, convenience sample — to real structural scenarios


In [ ]:
# Run this cell first — installs required libraries for interactive widgets
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '--quiet'])

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

# ── Build the population of 500 floor bays ───────────────────────────────────
# Live load values from Hibbeler Table 1-4 (ASCE 7-10 minimums)
#   Office space:      50 psf   (50% of building)
#   Assembly rooms:   100 psf   (20%)
#   Storage/warehouse: 125 psf  (20%)
#   Lobbies/corridors: 100 psf  (10%)
np.random.seed(42)
office   = np.random.normal(50,  6, 250)
assembly = np.random.normal(100, 10, 100)
storage  = np.random.normal(125, 15, 100)
lobby    = np.random.normal(100,  8,  50)
population  = np.concatenate([office, assembly, storage, lobby])
floor_types = np.array(['office']*250 + ['assembly']*100 + ['storage']*100 + ['lobby']*50)
shuffle_idx = np.random.permutation(500)
population  = population[shuffle_idx]
floor_types = floor_types[shuffle_idx]
TRUE_MEAN   = np.mean(population)
print(f'Building population: 500 floor bays | True mean load: {TRUE_MEAN:.1f} psf')


## 9.1  Why Sampling Matters in Structural Engineering

Before a structural engineer can design a building, bridge, or tower, they need to know what **loads** the structure must carry. Hibbeler's *Structural Analysis* (8th ed.) defines two fundamental load types (§1.3):

| Load Type | Definition | Example |
|-----------|-----------|--------|
| **Dead load** | Fixed weight of the structure itself | Beams, columns, floor slabs, roofing |
| **Live load** | Loads that can vary in magnitude and location | People, furniture, stored goods, vehicles |

Here is the challenge: **you can never measure every single load in a large structure.** A ten-story mixed-use building may have hundreds of floor bays, each with a different load depending on what occupies that space. Instead, engineers must:

1. Take a **sample** of load measurements
2. Use that sample to **estimate** the true load across the whole structure
3. Design the structure to handle that estimated load — plus a safety factor

> *But what if your sample gives you the wrong picture?*

Hibbeler notes (§1.3) that estimates of dead loading alone *"can be in error by 15% to 20% or more."* For a column carrying thousands of kips, a 15% error is not an abstraction — it can be the difference between a safe structure and a catastrophic failure.


## 🔬 Interactive Experiment 1: Does Your Sampling Method Matter?

Consider a **10-story mixed-use building** with 500 floor bays. Based on ASCE 7-10 minimum live loads (Hibbeler Table 1–4):

| Floor Type | Share of Building | Design Live Load |
|-----------|------------------|-----------------|
| Office space | 50 % | 50 psf |
| Assembly / meeting rooms | 20 % | 100 psf |
| Storage / warehouse | 20 % | 125 psf |
| Lobbies & corridors | 10 % | 100 psf |

**True average load across all 500 bays ≈ 80 psf.**

You are an inspector who needs to estimate this average. You have time to check **n** floor bays. Use the widget below to compare two approaches:

- **Simple Random Sample (SRS):** You randomly select bays from all areas of the building.
- **Convenience Sample:** You only inspect the easily-accessible lower-level office floors.


In [ ]:
def _sample_and_plot(method, n):
    fig, (ax_pop, ax_samp) = plt.subplots(1, 2, figsize=(14, 5))

    if method == 'Simple Random Sample (SRS)':
        idx    = np.random.choice(500, size=n, replace=False)
        sample = population[idx]
        color  = 'steelblue'
        label  = f'SRS  (n = {n})'
    else:
        office_idx = np.where(floor_types == 'office')[0]
        idx        = np.random.choice(office_idx, size=min(n, len(office_idx)), replace=False)
        sample     = population[idx]
        color      = 'firebrick'
        label      = f'Convenience — office floors only  (n = {n})'

    sample_mean = np.mean(sample)

    # Left: full population
    ax_pop.hist(population, bins=40, color='lightgray', edgecolor='white', alpha=0.85)
    ax_pop.axvline(TRUE_MEAN, color='black', lw=2, ls='--', label=f'True mean: {TRUE_MEAN:.1f} psf')
    ax_pop.set_xlabel('Live Load (psf)', fontsize=12)
    ax_pop.set_ylabel('Number of Floor Bays', fontsize=12)
    ax_pop.set_title('All 500 Floor Bays  (Population)', fontsize=13)
    ax_pop.legend(fontsize=11)

    # Right: sample overlaid
    ax_samp.hist(population, bins=40, color='lightgray', edgecolor='white', alpha=0.35)
    ax_samp.hist(sample, bins=20, color=color, edgecolor='white', alpha=0.75, label=label)
    ax_samp.axvline(TRUE_MEAN,   color='black', lw=2,   ls='--', label=f'True mean: {TRUE_MEAN:.1f} psf')
    ax_samp.axvline(sample_mean, color=color,   lw=2.5, ls='-',  label=f'Sample mean: {sample_mean:.1f} psf')
    ax_samp.set_xlabel('Live Load (psf)', fontsize=12)
    ax_samp.set_ylabel('Number of Floor Bays', fontsize=12)
    ax_samp.set_title('Your Sample', fontsize=13)
    ax_samp.legend(fontsize=10)

    if method != 'Simple Random Sample (SRS)':
        bias = sample_mean - TRUE_MEAN
        ax_samp.annotate(
            f'Bias: {bias:+.1f} psf\nUnderestimates true load!',
            xy=(0.05, 0.82), xycoords='axes fraction', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.85),
        )

    plt.tight_layout()
    plt.show()


w_method = widgets.Dropdown(
    options=['Simple Random Sample (SRS)', 'Convenience Sample (Office Floors Only)'],
    description='Method:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px'),
)
w_n = widgets.IntSlider(
    value=30, min=10, max=150, step=10,
    description='Sample size (n):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px'),
)
out1 = widgets.interactive_output(_sample_and_plot, {'method': w_method, 'n': w_n})
display(widgets.VBox([w_method, w_n, out1]))


### 💬 Stop and Think

1. When you use the **convenience sample**, does your load estimate get better as you increase *n*? Why or why not?
2. Which floor types does the convenience sample tend to *miss*? If a structural engineer used that estimate to size the building's columns, what could happen?
3. From Hibbeler Table 1–4: a heavy-storage warehouse floor carries **250 psf** — five times the office load. How would including just a few of those bays change the estimate?


---
## ⚠️  Real-World Consequence: The Hyatt Regency Walkway Collapse (1981)

On July 17, 1981, the atrium of the Kansas City Hyatt Regency Hotel held over 1,600 people watching a dance competition. Two suspended walkways — stacked one above the other — crossed the four-story atrium.

**The design change that killed 114 people:**

The original design used a single continuous rod running from the ceiling through both walkway support boxes. During construction, the contractor changed this to *two separate rods* — one from the ceiling to the 4th-floor box, and a second from the 4th-floor box down to the 2nd-floor box.

This seemed like a minor revision. But the load on the 4th-floor support box **doubled** — because it now carried the weight of *both* walkways instead of just routing the rod.

**The sampling parallel:**

The engineers who approved the change checked the same load they had always analyzed: the weight of a single walkway. Their load analysis was like a **convenience sample** — they examined the part of the load path that was familiar and easy to calculate, and never re-sampled the full load state that the new configuration created.

The connection failed at **60% of the required design load.** The walkways collapsed in seconds.

> *"No matter how carefully you analyze the loads you know about, if your model systematically misses a key part of the picture, your conclusions will be catastrophically wrong."*

This is not a math failure. It is a **sampling failure**: the engineers' mental model of the loads was an unrepresentative sample of the true load state.


---
## 9.2  Parameters and Statistics

In both data science and structural engineering, we distinguish carefully between two types of numbers:

| Term | Definition | Structural Example |
|------|-----------|-------------------|
| **Parameter** | A number that describes the *entire population* | True average live load across all 500 floor bays: **80 psf** |
| **Statistic** | A number computed from a *sample* | Average load from the 30 bays you actually inspected |

The ASCE 7-10 minimum live loads in Hibbeler Table 1–4 (50 psf for offices, 125 psf for light storage, etc.) are themselves **parameters** — values derived from decades of load surveys across thousands of buildings worldwide.

When an engineer measures actual loads in *your specific building* and computes an average, that is a **statistic** — an estimate of the parameter.

**Key insight:** Statistics are always uncertain. The goal of good sampling design is to make that uncertainty as small as possible — and to make sure the uncertainty is *random* rather than *systematic* (biased).


---
## 🔬 Interactive Experiment 2: How Much Does Sample Size Matter?

The widget below takes **200 repeated random samples** of size *n* from our 500-bay building and shows the distribution of resulting load estimates.

- **Small n:** Estimates vary widely — some could be dangerously low, others unrealistically high
- **Large n:** Estimates cluster tightly around the true mean of ~80 psf

Move the slider and watch the spread of estimates shrink. This is why codes require inspection of *enough* floor bays to get a reliable estimate — not just the first few you can find.


In [ ]:
def _sample_size_plot(n):
    sample_means = [np.mean(np.random.choice(population, size=n, replace=False)) for _ in range(200)]
    spread = np.std(sample_means)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(sample_means, bins=30, color='steelblue', edgecolor='white', alpha=0.75)
    ax.axvline(TRUE_MEAN,            color='black',     lw=2,   ls='--', label=f'True mean: {TRUE_MEAN:.1f} psf')
    ax.axvline(np.mean(sample_means), color='steelblue', lw=2.5, ls='-',
               label=f'Avg of 200 sample means: {np.mean(sample_means):.1f} psf')
    ax.set_xlabel('Estimated Mean Load (psf)', fontsize=12)
    ax.set_ylabel('Frequency  (out of 200 samples)', fontsize=12)
    ax.set_title(f'Spread of Load Estimates from 200 Random Samples  (n = {n})', fontsize=13)
    ax.legend(fontsize=11)
    ax.annotate(
        f'Estimate spread: ±{spread:.1f} psf',
        xy=(0.67, 0.86), xycoords='axes fraction', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85),
    )
    plt.tight_layout()
    plt.show()


w_n2 = widgets.IntSlider(
    value=10, min=5, max=200, step=5,
    description='Sample size (n):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px'),
)
out2 = widgets.interactive_output(_sample_size_plot, {'n': w_n2})
display(widgets.VBox([w_n2, out2]))


---
## 9.3  Sampling Methods in Structural Inspection

Structural engineers use several sampling strategies depending on the structure. These match the same methods used in statistics:

| Sampling Method | How It Works | Structural Example |
|----------------|-------------|-------------------|
| **Simple Random Sample** | Every element has equal probability of selection | Randomly selecting floor bays from a full building roster |
| **Systematic Sample** | Select every *k*th element in a sequence | Inspecting every 10th weld along a steel girder |
| **Stratified Sample** | Divide into groups (strata), then randomly sample within each | Separate office, storage, and assembly floors into strata, then randomly sample each |
| **Cluster Sample** | Randomly select entire sections, then inspect everything in them | Randomly pick 3 floors and inspect all bays on those floors |
| **Census** | Inspect every single element | Used for safety-critical connections in nuclear containment structures |

**Why stratified sampling is often best:** If you use SRS on our 500-bay building, there is a small but non-zero chance your sample contains no storage bays at all — and those carry the highest loads. Stratified sampling *guarantees* that each floor type is represented.


---
## 9.4  Example: Multistage Sampling for National Bridge Inspection

The Federal Highway Administration (FHWA) must inspect roughly **600,000 bridges** across the United States. Inspecting every element of every bridge is impossible. Instead, a four-stage sampling strategy is used:

**Stage 1 — Stratify by condition rating**  
Bridges are divided by their last inspection score: Good / Fair / Poor. Poor-condition bridges are sampled at a higher rate.

**Stage 2 — Select state DOT districts**  
Within each stratum, randomly choose state transportation districts to visit.

**Stage 3 — Select bridge types within each district**  
Group bridges by structural type (truss, girder, cable-stayed, arch) and systematically select from each group.

**Stage 4 — Inspect individual elements**  
Within each selected bridge, randomly choose connections, welds, and members for detailed load and stress testing.

This four-stage design ensures the *sample* of inspected bridges is representative of the full *population* — including the rare but critical high-load and poor-condition cases that convenience sampling would miss.


---
## 📋  Chapter 9 Review

**Vocabulary:**

| Term | Meaning |
|------|--------|
| **Population** | All structural elements or load cases of interest |
| **Sample** | The subset you actually measure |
| **Parameter** | A number describing the entire population |
| **Statistic** | A number computed from a sample (an estimate of the parameter) |
| **Sampling Bias** | When the sampling method systematically favors certain values |
| **SRS** | Simple Random Sample — every element has an equal chance of selection |
| **Convenience Sample** | Selecting what is easy to reach — almost always biased |

**The Big Idea:**  
Structural failures often happen not because engineers made arithmetic errors, but because their *model of the loads* was an incomplete, biased sample of reality. Whether it is an inspector who only checks the easy floors, a wind tunnel that only tests calm conditions, or a design review that only analyzes one load path — biased sampling means even perfect calculations give wrong answers.

Representative sampling is not just a statistics concept. In structural engineering, it is a matter of life and safety.
